In [0]:
#setup 
from pyspark.sql import functions as F

PRICING = "alpenia_dev.bronze.brz_pricing"
EVENTS  = "alpenia_dev.bronze.brz_web_events"

def history(table, n=10):
    display(
        spark.sql(f"DESCRIBE HISTORY {table}")
        .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
        .orderBy(F.desc("version"))
        .limit(n)
    )

def last_version(table):
    return spark.sql(f"DESCRIBE HISTORY {table}").agg(F.max("version")).first()[0]

# Time Travel and RESTORE on brz_pricing

In [0]:
# Before
v_before = last_version(PRICING)
print("Last good version:", v_before)
display(spark.table(PRICING).orderBy("city", "product"))

In [0]:
%sql
-- bad update
UPDATE alpenia_dev.bronze.brz_pricing
SET base_premium_eur = base_premium_eur * 2
WHERE city = 'Paris';

In [0]:
# after
print("CURRENT (wrong):")
display(spark.sql(f"SELECT * FROM {PRICING} ORDER BY city, product"))

print(f"VERSION AS OF {v_before} (correct):")
display(spark.sql(f"SELECT * FROM {PRICING} VERSION AS OF {v_before} ORDER BY city, product"))

In [0]:
# RESTORE
spark.sql(f"RESTORE TABLE {PRICING} TO VERSION AS OF {v_before}")

# Proof: the table is identical to the good version, in both directions
now = spark.table(PRICING)
old = spark.sql(f"SELECT * FROM {PRICING} VERSION AS OF {v_before}")
assert now.exceptAll(old).count() == 0 and old.exceptAll(now).count() == 0
print("Restore OK: content identical to version", v_before)

In [0]:
history(PRICING)
# Expected: WRITE (creation) -> WRITE (typing in 03) -> UPDATE -> RESTORE
# RESTORE adds a NEW version; it does not delete the UPDATE version.

# OPTIMIZE on brz_web_events

In [0]:
# Before
def detail(table):
    return (spark.sql(f"DESCRIBE DETAIL {table}")
            .select("numFiles", "sizeInBytes").first())

d_before = detail(EVENTS)
print(f"BEFORE: {d_before.numFiles} files, {d_before.sizeInBytes:,} bytes")

In [0]:
%sql
OPTIMIZE alpenia_dev.bronze.brz_web_events;

In [0]:
d_after = detail(EVENTS)
print(f"AFTER : {d_after.numFiles} files, {d_after.sizeInBytes:,} bytes")

display(
    spark.sql(f"DESCRIBE HISTORY {EVENTS}")
    .filter("operation = 'OPTIMIZE'")
    .select("version", "operation",
            F.col("operationMetrics.numRemovedFiles").alias("files_removed"),
            F.col("operationMetrics.numAddedFiles").alias("files_added"))
)

* OPTIMIZE compacts many small files into fewer large ones. Fewer files means less metadata to read and faster scans.
* The old files are not physically deleted. They are marked as removed in the log, so time travel still works. Only VACUUM deletes them.
* The drop in numFiles depends on how many small files the write left behind. If the count was already 1 or 2, the gap will be small. Databricks can also compact automatically on managed tables, so a small gap is not a mistake.

# VACUUM

In [0]:
%sql
VACUUM alpenia_dev.bronze.brz_pricing DRY RUN;